In [1]:
from ultralytics import YOLO
from pathlib import Path
from datetime import datetime

# โหลดโมเดลที่ train แล้ว (เปลี่ยน path ให้ตรงกับ best.pt ล่าสุดของคุณ)
model = YOLO(r"D:\move From SSD\python_test - Test GPU CUDA\runs\detect\train-2\weights\best.pt")

# ชี้ไปที่ไฟล์ data.yaml ของชุดข้อมูล
path = Path(r"D:\move From SSD\python_test - Test GPU CUDA\CDP.v2i.yolov8\data.yaml")

# ชื่อคลาสต้องตรงกับใน data.yaml ทุกตัวอักษร
class_name = ['cat', 'dog', 'panda']

# กำหนดชุดทดสอบ: 1 แถว = 1 รูปที่ใช้ให้คะแนน
# - image: ชื่อไฟล์รูป
# - correct_class: คลาสที่รูปนั้นควรถูกทำนาย
# - expected_count: จำนวน object ที่คาดหวังในรูป
TEST_CASES = [
    {"image": "cat1.png", "correct_class": "cat", "expected_count": 1},
    {"image": "cat2.png", "correct_class": "cat", "expected_count": 2},
    {"image": "dog1.png", "correct_class": "dog", "expected_count": 1},
    {"image": "dog3.png", "correct_class": "dog", "expected_count": 3},
    {"image": "panda3.jpg", "correct_class": "panda", "expected_count": 3},
    {"image": "panda2.jpg", "correct_class": "panda", "expected_count": 2},
]


def detect_box(cases):
    # คงค่าหักคะแนนต่อกล่องผิดเหมือนเดิม
    PENALTY_WRONG_PER_BOX = 3.0

    # ปรับตามจำนวนรูป เพื่อให้คะแนน Detect-Box เต็มรวม = 100 เสมอ
    # (ถ้าไม่ปรับ เมื่อจำนวนรูปไม่ใช่ 4 คะแนนเต็มรวมจะไม่เท่ากับ 100)
    IMAGE_SCORE_MAX = 100.0 / len(cases) if cases else 0.0

    def score_one_image(img_path: str, correct_cls: str, expected_count: int) -> float:
        conf_correct = []
        conf_wrong = []

        # รันตรวจจับบนรูป 1 รูป
        results = model(img_path, save=True, conf=0.1) #conf=0.5 คือกำหนดความมั่นใจของกล่องที่จะถูกต้อง

        # เก็บ confidence แยกเป็น "ทายถูกคลาส" และ "ทายผิดคลาส"
        # สูตรคิดคะแนนหลักคงเดิม: ใช้ conf ของคลาสถูก + หักกล่องผิด
        for result in results:
            for box in result.boxes:
                cls_id = int(box.cls[0])
                pred_cls = model.names[cls_id]
                conf = float(box.conf[0])

                if pred_cls == correct_cls:
                    conf_correct.append(conf)
                else:
                    conf_wrong.append(conf)

        k = expected_count
        conf_correct_sorted = sorted(conf_correct, reverse=True)
        keep = conf_correct_sorted[:k] if k > 0 else []
        n_correct = len(conf_correct)

        # คงสูตรเดิม:
        # base = ค่าเฉลี่ย conf ของ top-k กล่องที่ถูกคลาส
        base = (sum(keep) / k) if k > 0 else 0.0

        # คงสูตรเดิม:
        # ถ้าทายถูกคลาส "เกินจำนวนที่คาดหวัง" จะถูกลดด้วย over_factor
        over_factor = (k / n_correct) if (k > 0 and n_correct > k) else 1.0

        # คงโครงสูตรเดิม: คะแนนดิบ - ค่าปรับกล่องผิด
        raw_score = IMAGE_SCORE_MAX * base * over_factor
        wrong_penalty = PENALTY_WRONG_PER_BOX * len(conf_wrong)
        image_score = max(0.0, raw_score - wrong_penalty)

        # พิมพ์รายละเอียดรายรูป
        print(f"\nImage: {img_path}")
        print(f"  class={correct_cls}, expected_count={k}, n_correct={n_correct}, n_wrong={len(conf_wrong)}")
        print(f"  top-{k} correct conf={[round(x, 3) for x in keep]}")
        print(f"  score={image_score:.2f} / {IMAGE_SCORE_MAX:.2f}")

        return image_score

    # เก็บคะแนนรวมรายคลาส และคะแนนรวมทั้งหมด
    per_class_score = {name: 0.0 for name in class_name}
    total_score = 0.0

    for case in cases:
        score = score_one_image(case["image"], case["correct_class"], case["expected_count"])
        per_class_score[case["correct_class"]] = per_class_score.get(case["correct_class"], 0.0) + score
        total_score += score

    # สรุปคะแนน Detect-Box
    print("\nPer-class Detect-Box score")
    for cls, score in per_class_score.items():
        print(f"  {cls}: {score:.2f}")

    print(f"FINAL TOTAL DETECT-BOX SCORE: {total_score:.2f} / 100")
    return total_score, per_class_score


if __name__ == '__main__':
    now = datetime.now().strftime("%d-%m-%Y - %H:%M:%S")
    print(f"\n===== RUN STARTED AT {now} =====\n")

    # วัด mAP จากชุด val
    metrics = model.val(data=path, imgsz=640)
    map50 = metrics.box.map50 * 100
    map75 = metrics.box.map75 * 100

    # วัดคะแนน Detect-Box จากรูป test ที่กำหนดใน TEST_CASES
    detect_box_score, per_class_scores = detect_box(TEST_CASES)
    total_score = map50 + map75 + detect_box_score

    # สรุปคะแนนรวมทั้งหมด
    print("********** Final Summary Score **********")
    print(f"Timestamp:     {now}")
    print(f"mAP50:         {map50:.2f}")
    print(f"mAP75:         {map75:.2f}")
    print(f"Detect-Box:    {detect_box_score:.2f}")
    for cls, score in per_class_scores.items():
        print(f"Detect-Box ({cls}): {score:.2f}")
    print(f"Total Score:   {total_score:.2f}")


===== RUN STARTED AT 24-04-2026 - 13:53:33 =====

Ultralytics 8.4.41  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 Ti with Max-Q Design, 6144MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 69.39.8 MB/s, size: 41.8 KB)
val: Scanning D:\move From SSD\python_test - Test GPU CUDA\CDP.v2i.yolov8\valid\labels.cache... 11 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11/11  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 6.0s/it 6.0s
                   all         11         12      0.882      0.667      0.987      0.876
                   cat          5          6      0.792          1      0.972      0.688
                   dog          2          2          1          0      0.995      0.945
                 panda          4          4      0.855          1      0.995      0.995
Speed: 6.5ms preprocess, 30.2ms 